# Маркет ВКонтакте и партнёр: итоги и роль Маркета

Период — 30 дней сентября 2026 года. Сравниваются агрегаты Маркета VK и условного внешнего партнёра; пути заказа могут различаться. Источник синтетический; показатели каналов и их пользователей могут пересекаться. Для повторного запуска откройте ноутбук из папки `notebooks`: CSV ожидаются в соседней папке `Data`.

**Важно:** прямые переходы у Маркета и оплаты у партнёра в исходных данных заполнены нулём как неприменимые показатели; эти две ячейки переводятся в пропуски, остальные нули сохраняются.

## Как читать маршруты: уточнение для анализа

Ниже используется **рабочая трактовка, уточнённая командой**, а не доказанное по выгрузке техническое определение каждого события:

- **Маркет VK:** открытие карточки товара, относящегося к Маркету, во VK → нажатие кнопки → возможный переход к чекауту VK или в чат с продавцом. Кейс упоминает и внешние сайты; фактические назначения кнопок в выгрузке не разделены.
- **Партнёр 1 через карточку VK:** открытие карточки его товара во VK → нажатие кнопки → переход к чекауту на сайте партнёра (по уточнённому сценарию). Это ещё не оформленный заказ.
- **Партнёр 1 напрямую:** переход из VK без открытия карточки во VK → открытие страницы товара на сайте партнёра (по уточнённому сценарию); в данных этот маршрут отражён как прямые переходы, а не как открытия карточки VK.

Партнёр 1 — условная площадка, не обязательно Ozon. Товар Маркета может продаваться и на других площадках. Назначения кнопок и факт открытия страницы партнёра нужно подтвердить по справочнику событий и интеграции; эти детали отдельно в CSV не измерены. Прямые переходы и нажатия ведут на разные шаги и могут пересекаться по пользователям: их сумма не является числом уникальных покупателей или единым знаменателем конверсии. Сравнение агрегированных отношений — не когортная конверсия и не диагностика причин.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path.cwd().parent / 'Data'
mv = 'Маркет Вконтакте'
p1 = 'Партнёр 1'
pd.set_option('display.float_format', '{:,.2f}'.format)

market = pd.read_csv(data_path / 'Market_Partner.csv').set_index('Метрика')
total = pd.read_csv(data_path / 'Market_Total.csv').set_index('Период')
market.loc[['Прямые переходы к Партнёру 1 (без открытия карточки)'], mv] = market.loc[['Прямые переходы к Партнёру 1 (без открытия карточки)'], mv].replace(0, np.nan)
market.loc[['Оплаченные заказы'], p1] = market.loc[['Оплаченные заказы'], p1].replace(0, np.nan)
market

,Маркет Вконтакте,Партнёр 1
Метрика,,
Показы товаров,"4,777,750,023.00","5,729,628,790.00"
Открытия карточки товара,"193,512,461.00","86,882,822.00"
Нажатия на кнопку в карточке,"5,969,063.00","36,626,075.00"
Прямые переходы к Партнёру 1 (без открытия карточки),NaN,"106,914,679.00"
Оформленные заказы,"413,740.00","4,927,525.00"
Оплаченные заказы,"4,903.00",NaN
"GMV, ₽","1,564,546,740.00","5,119,514,685.00"
"Средний чек, ₽","3,781.00","1,039.00"
Уникальные покупатели за 30 дней,"296,148.00","2,901,334.00"


In [ ]:
# Интерактивные визуализации
import plotly.graph_objects as go

def show_plotly(fig, title, x_title, y_title, height=500):
    fig.update_layout(
        title=dict(text=title), template='plotly_white', height=height,
        xaxis_title=x_title, yaxis_title=y_title,
        legend_title_text='Канал', margin=dict(l=70, r=30, t=80, b=90),
        font=dict(size=13), hoverlabel=dict(namelength=-1),
    )
    fig.show(config={'displaylogo': False, 'responsive': True})


## Проверка производных показателей

Сопоставляем приведённые в таблице показатели с пересчётом из абсолютных значений. В исходной таблице показатели округлены. Пропуски означают отсутствие сопоставимого значения, а не нулевой результат.

In [2]:
derived = pd.DataFrame(index=['Средний чек, ₽', 'Заказов на покупателя', 'Конверсия: показ → открытие карточки, %', 'Конверсия: открытие карточки → заказ, %', 'Конверсия: показ → заказ, %'], columns=[mv, p1], dtype=float)
derived.loc['Средний чек, ₽'] = market.loc['GMV, ₽'] / market.loc['Оформленные заказы']
derived.loc['Заказов на покупателя'] = market.loc['Оформленные заказы'] / market.loc['Уникальные покупатели за 30 дней']
derived.loc['Конверсия: показ → открытие карточки, %'] = market.loc['Открытия карточки товара'] / market.loc['Показы товаров'] * 100
derived.loc['Конверсия: открытие карточки → заказ, %'] = market.loc['Оформленные заказы'] / market.loc['Открытия карточки товара'] * 100
derived.loc['Конверсия: показ → заказ, %'] = market.loc['Оформленные заказы'] / market.loc['Показы товаров'] * 100
pd.concat({'В данных': market.loc[derived.index], 'Пересчёт': derived}, axis=1)

В данных  ...          
                                        Маркет Вконтакте  ... Партнёр 1
Средний чек, ₽                                  3,781.00  ...  1,038.96
Заказов на покупателя                               1.40  ...      1.70
Конверсия: показ → открытие карточки, %             4.05  ...      1.52
Конверсия: открытие карточки → заказ, %             0.21  ...      5.67
Конверсия: показ → заказ, %                         0.01  ...      0.09

[5 rows x 4 columns]

## Масштаб и эффективность

Доля Маркета — его показатель, делённый на сумму показателей двух каналов, а не доля уникального рынка. Воронки не являются строгими когортными: один человек может совершить несколько действий. У партнёра заказы могут приходить и из прямых переходов к странице товара вне VK, в обход карточки VK, поэтому отношение заказов к открытиям не является чистой конверсией одной и той же когорты.

In [3]:
volume_metrics = ['Показы товаров', 'Открытия карточки товара', 'Нажатия на кнопку в карточке', 'Оформленные заказы', 'GMV, ₽', 'Уникальные покупатели за 30 дней']
volumes = market.loc[volume_metrics].copy()
volumes['Доля Маркета в сумме каналов, %'] = volumes[mv] / (volumes[mv] + volumes[p1]) * 100
volumes

,Маркет Вконтакте,Партнёр 1,"Доля Маркета в сумме каналов, %"
Метрика,,,
Показы товаров,"4,777,750,023.00","5,729,628,790.00",45.47
Открытия карточки товара,"193,512,461.00","86,882,822.00",69.01
Нажатия на кнопку в карточке,"5,969,063.00","36,626,075.00",14.01
Оформленные заказы,"413,740.00","4,927,525.00",7.75
"GMV, ₽","1,564,546,740.00","5,119,514,685.00",23.41
Уникальные покупатели за 30 дней,"296,148.00","2,901,334.00",9.26


### Интерактивно: доля Маркета в сумме двух каналов
Показаны разные показатели в одинаковой относительной шкале. Покупатели каналов могут пересекаться; сумма не является числом уникальных людей или размером рынка.


In [ ]:
fig = go.Figure(go.Bar(
    x=volumes['Доля Маркета в сумме каналов, %'].iloc[::-1],
    y=volumes.index[::-1], orientation='h', marker_color='#3978C6',
    customdata=volumes[[mv, p1]].iloc[::-1].to_numpy(),
    hovertemplate='%{y}<br>Доля Маркета: %{x:.2f}%<br>Маркет: %{customdata[0]:,.0f}<br>Партнёр: %{customdata[1]:,.0f}<extra></extra>'
))
fig.update_xaxes(range=[0, 100], ticksuffix='%')
show_plotly(fig, 'Масштаб Маркета относительно суммы каналов', 'Доля Маркета в сумме двух каналов, %', '', 500)


Интерактивный график Plotly: Масштаб Маркета относительно суммы каналов

In [4]:
efficiency = pd.DataFrame(index=['Открытия / показы, %', 'Нажатия / открытия, %', 'Заказы / открытия, %', 'Заказы / показы, %', 'GMV на 1 000 показов, ₽', 'GMV на открытие, ₽', 'GMV на покупателя, ₽', 'Заказов на покупателя', 'Повторные покупатели, %'], columns=[mv, p1], dtype=float)
efficiency.loc['Открытия / показы, %'] = market.loc['Открытия карточки товара'] / market.loc['Показы товаров'] * 100
efficiency.loc['Нажатия / открытия, %'] = market.loc['Нажатия на кнопку в карточке'] / market.loc['Открытия карточки товара'] * 100
efficiency.loc['Заказы / открытия, %'] = market.loc['Оформленные заказы'] / market.loc['Открытия карточки товара'] * 100
efficiency.loc['Заказы / показы, %'] = market.loc['Оформленные заказы'] / market.loc['Показы товаров'] * 100
efficiency.loc['GMV на 1 000 показов, ₽'] = market.loc['GMV, ₽'] / market.loc['Показы товаров'] * 1000
efficiency.loc['GMV на открытие, ₽'] = market.loc['GMV, ₽'] / market.loc['Открытия карточки товара']
efficiency.loc['GMV на покупателя, ₽'] = market.loc['GMV, ₽'] / market.loc['Уникальные покупатели за 30 дней']
efficiency.loc['Заказов на покупателя'] = market.loc['Оформленные заказы'] / market.loc['Уникальные покупатели за 30 дней']
efficiency.loc['Повторные покупатели, %'] = market.loc['Доля повторных покупателей (2+ заказа), %']
efficiency

,Маркет Вконтакте,Партнёр 1
"Открытия / показы, %",4.05,1.52
"Нажатия / открытия, %",3.08,42.16
"Заказы / открытия, %",0.21,5.67
"Заказы / показы, %",0.01,0.09
"GMV на 1 000 показов, ₽",327.47,893.52
"GMV на открытие, ₽",8.08,58.92
"GMV на покупателя, ₽","5,282.99","1,764.54"
Заказов на покупателя,1.40,1.70
"Повторные покупатели, %",19.00,26.60


### Интерактивно: отношения событий
Это отношения агрегированных событий, а не когортные конверсии. Заказы партнёра включают прямые переходы, в том числе без открытия карточки.

Различие 3,08% и 42,16% не означает сопоставимую конверсию: кнопки ведут к разным следующим шагам. Разбивки по назначению и связанной когорты нет.


In [ ]:
rate_names = ['Открытия / показы, %', 'Нажатия / открытия, %', 'Заказы / показы, %']
fig = go.Figure()
for channel, color in [(mv, '#3978C6'), (p1, '#F2994A')]:
    fig.add_bar(name=channel, x=rate_names, y=efficiency.loc[rate_names, channel], marker_color=color,
                hovertemplate='%{x}<br>%{fullData.name}: %{y:.2f}%<extra></extra>')
fig.update_layout(barmode='group')
fig.update_yaxes(rangemode='tozero', ticksuffix='%')
show_plotly(fig, 'Сопоставление отношений событий', '', 'Отношение событий, %', 460)


Интерактивный график Plotly: Сопоставление отношений событий

In [5]:
direct = market.loc['Прямые переходы к Партнёру 1 (без открытия карточки)', p1]
routes = pd.Series({
    'Партнёр: нажатия на кнопку карточки VK': market.loc['Нажатия на кнопку в карточке', p1],
    'Партнёр: прямые переходы без карточки VK': direct,
    'Маркет: нажатия на кнопку карточки VK': market.loc['Нажатия на кнопку в карточке', mv],
}, name='Событий за 30 дней')
routes


,Событий за 30 дней
Партнёр: нажатия на кнопку карточки VK,36626075
Партнёр: прямые переходы без карточки VK,106914679
Маркет: нажатия на кнопку карточки VK,5969063


### Выводы

- На Маркет приходится около 69% открытий карточек двух каналов, но 7,7% оформленных заказов и 23,4% GMV. Средний чек Маркета около 3 781 ₽ против 1 039 ₽ у партнёра.
- Нажатия относительно открытий — 3,1% у Маркета и 42,2% у партнёра. Кнопки ведут к разным следующим шагам, а прямой маршрут партнёра обходит карточку VK. Эти отношения нельзя считать сопоставимыми конверсиями или доказательством проблем чекаута.
- 4 903 оплаты составляют 1,19% оформленных заказов Маркета. Это не доказывает, что остальные заказы не были оплачены вне ВК. Сначала проверить учёт оплат и маршруты; проверка условий чекаута — гипотеза, а не диагноз.
- На одного покупателя Маркет генерирует больше GMV, однако покупателей и заказов существенно меньше. При партнёрской модели часть покупок происходит вне ВК; сравнение не измеряет все внешние продажи бизнеса.

## Охват отдельного Маркета

MAU отдельной витрины, увидевшие товары и открывшие карточку относятся к разным, возможно пересекающимся аудиториям. Соотношение ниже — масштаб относительно охвата, не доказанная доля одной когорты.

In [6]:
total


,Значение
Период,
DAU Маркета Вконтакте (среднее за день),"127,406.00"
MAU Маркета Вконтакте (уникальные за 30 дней),"2,997,341.00"
"Пользователи, видевшие товары за 30 дней","66,617,967.00"
"Пользователи, открывавшие карточку товара за 30 дней","27,333,960.00"
Открытия карточки товара из постов и клипов,"86,321,893.00"
в т.ч. из реферальных постов и клипов,"82,324,081.00"
Запросы в товарном поиске,"28,585,243.00"
"Доля поисковых запросов с открытием карточки, %",18.40


In [7]:
reach = pd.Series({'MAU Маркета / видевшие товары, %': total.loc['MAU Маркета Вконтакте (уникальные за 30 дней)', 'Значение'] / total.loc['Пользователи, видевшие товары за 30 дней', 'Значение'] * 100, 'Открывавшие / видевшие товары, %': total.loc['Пользователи, открывавшие карточку товара за 30 дней', 'Значение'] / total.loc['Пользователи, видевшие товары за 30 дней', 'Значение'] * 100, 'Средний DAU / MAU, %': total.loc['DAU Маркета Вконтакте (среднее за день)', 'Значение'] / total.loc['MAU Маркета Вконтакте (уникальные за 30 дней)', 'Значение'] * 100})
reach

MAU Маркета / видевшие товары, %    4.50
Открывавшие / видевшие товары, %   41.03
Средний DAU / MAU, %                4.25
dtype: float64

MAU отдельного Маркета — около 3 млн против 66,6 млн пользователей, видевших товары. В сочетании с малой долей открытий с главной Маркета (см. ноутбук 03) это поддерживает гипотезу об инфраструктурной роли каталога, поиска, карточек и чекаута внутри ВК, а не о ставке только на отдельную витрину. Доказать, что витрина не нужна, эти агрегаты не позволяют.